# Profiling for PATH

In [1]:
# imports
from importlib import reload
import os
from importlib.resources import files as resource_files

import numpy as np

import pandas

from astropy.coordinates import SkyCoord
from astropy.coordinates import offset_by
from astropy import units
#from astropy.io import fits

from astropath import path
from astropath import localization
from astropath import bayesian

# Convenience class

In [2]:
Path = path.PATH()

# Faux FRB

In [3]:
## FRB Coord
frb_coord = SkyCoord('21h44m25.255s -40d54m00.10s', frame='icrs')
frb_coord

<SkyCoord (ICRS): (ra, dec) in deg
    (326.10522917, -40.90002778)>

# Prior

In [4]:
theta_prior = dict(max=6., PDF='exp', scale=1.)

# Localization

In [5]:
# Small, but not tiny
eellipse = dict(a=5, b=5, theta=0.)

## Build it

In [6]:
Path.init_localization('eellipse', center_coord=frb_coord, eellipse=eellipse)

In [7]:
Path.localiz

{'type': 'eellipse',
 'center_coord': <SkyCoord (ICRS): (ra, dec) in deg
     (326.10522917, -40.90002778)>,
 'eellipse': {'a': 5, 'b': 5, 'theta': 0.0}}

# Galaxies

## Positions

In [8]:
# 1.0" North, 1.0" East
gal_coord1 = frb_coord.directional_offset_by(0.*units.deg, 1.0*units.arcsec)
gal_coord2 = frb_coord.directional_offset_by(90.*units.deg, 1.0*units.arcsec)

# Sizes

In [9]:
small_gal_size = np.array([1.]) # arcsec
tiny_gal_size = np.array([0.2]) # arcsec
box_hwidth = 50.

In [14]:
box_hwidth = 90. # arcsec
step_size = 0.5 / 20 # arcsize

# Calculate

In [13]:
# Set Equinox (for spherical offsets)
localiz = Path.localiz.copy()
localiz['center_coord'].equinox = gal_coord1.equinox

## Init the main grid :: $\sim 0.1$s

In [17]:
%%timeit
# Build the fixed grid around the transient
ngrid = int(np.round(2*box_hwidth / step_size))
x = np.linspace(-box_hwidth, box_hwidth, ngrid)
xcoord, ycoord = np.meshgrid(x,x)

# Grid spacing
grid_spacing_arcsec = x[1]-x[0]

125 ms ± 3.72 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## $L(w,x)$ ::  $\sim 10$s

In [22]:
%%timeit
# #####################
# L(w-x) -- 2D Gaussian, normalized to 1 when integrating over x not omega
# Approximate as flat sky
#  Warning:  RA increases in x for these grids!!
ra = localiz['center_coord'].ra.deg + \
    xcoord/3600. / np.cos(localiz['center_coord'].dec).value
dec = localiz['center_coord'].dec.deg + ycoord/3600.
L_wx = localization.calc_LWx(ra, dec, localiz)

8.48 s ± 113 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Offsets :: $\sim 0.5$s

In [24]:
%%timeit
# Offsets from the transient (approximate + flat sky)
cand_coord = gal_coord1
theta = 3600*np.sqrt(np.cos(cand_coord.dec).value**2 * (
    ra-cand_coord.ra.deg)**2 + (dec-cand_coord.dec.deg)**2)  # arc sec

474 ms ± 6.39 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## $p(w,O_i)$ :: $\sim 0.1$s

### This step could be sped up by cutting out a portion of the full grid

In [26]:
%%timeit
# p(w|O_i)
p_wOi = bayesian.pw_Oi(theta,
      0.2, # arcsec
      theta_prior)

110 ms ± 3.15 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Product :: $\sim 0.1$s

In [28]:
%%timeit
# Product
grid_p = L_wx * p_wOi

109 ms ± 782 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Sum :: $\sim 0.3$s

In [31]:
%%timeit
tsum = np.sum(grid_p)*grid_spacing_arcsec**2

26.3 ms ± 552 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
